# 06 - Prepare the current dataset for K-Means

Input: the processed `_144.csv` dataset. Exclude Protected stations, select annual
erosion rate, physical erosion score and economic score, then standardize the
three features. This notebook is exploratory preparation; Notebook 07 rebuilds
the scaler in its own kernel and saves it together with the final K-Means model.


In [1]:
from pathlib import Path

# Locate the project from either the repository root or its notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data" / "raw" / "jps_coastal_erosion_full.csv").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Open this notebook from inside FYP_Coastal_Erosion.")
    PROJECT_ROOT = PROJECT_ROOT.parent

import pandas as pd
import numpy as np

# Load updated processed dataset
df = pd.read_csv(
    PROJECT_ROOT / "data/processed/terengganu_coastal_erosion_transects_144.csv"
)

# Exclude protected locations from K-Means training
df_model = df[df["Is_Protected"] == 0].copy()

print("Total points:", len(df))
print("Points for K-Means:", len(df_model))
print("Protected points excluded:", df["Is_Protected"].sum())


Total points: 2775
Points for K-Means: 2766
Protected points excluded: 9


In [2]:
# Step 2: Feature selection for K-Means

features = [
    "Annual_Erosion_Rate_m_yr",
    "Physical_Erosion_Score",
    "Economic_Score"
]

# Select features
X = df_model[features].copy()

print("=== FEATURE SELECTION ===")
print("Selected features:", features)
print("Feature matrix shape:", X.shape)

print("\nMissing values:")
print(X.isnull().sum())

print("\nTraining data sources:")
print(df_model["Source_Type"].value_counts())

print("\nFirst 5 rows:")
display(X.head())

assert X.notna().all().all(), "Missing K-Means inputs"
assert set(df_model["Source_Type"]).issubset({"JPS_Original", "Interpolated"})


=== FEATURE SELECTION ===
Selected features: ['Annual_Erosion_Rate_m_yr', 'Physical_Erosion_Score', 'Economic_Score']
Feature matrix shape: (2766, 3)

Missing values:
Annual_Erosion_Rate_m_yr    0
Physical_Erosion_Score      0
Economic_Score              0
dtype: int64

Training data sources:
Source_Type
Interpolated    2631
JPS_Original     135
Name: count, dtype: int64

First 5 rows:


,Annual_Erosion_Rate_m_yr,Physical_Erosion_Score,Economic_Score
0,4.9400,5.0000,3.4000
1,4.8632,4.9223,3.3969
2,4.7864,4.8446,3.3938
3,4.7097,4.7669,3.3907
4,4.6329,4.6892,3.3876


In [3]:
# Step 3: Feature scaling

from sklearn.preprocessing import StandardScaler

# Initialize scaler
scaler = StandardScaler()

# Standardize the selected features
X_scaled = scaler.fit_transform(X)

print("=== FEATURE SCALING ===")

print("Original feature shape:", X.shape)
print("Scaled feature shape:", X_scaled.shape)

print("\nScaled feature means:")
print(X_scaled.mean(axis=0).round(4))

print("\nScaled feature standard deviations:")
print(X_scaled.std(axis=0).round(4))

print("\nFirst 5 scaled rows:")
print(X_scaled[:5].round(4))

X_scaled_df = pd.DataFrame(X_scaled, columns=features, index=X.index)
print("\nK-Means preparation completed.")
print("Number of samples:", len(X_scaled_df))
print("Number of features:", len(features))
display(X_scaled_df.describe())


=== FEATURE SCALING ===
Original feature shape: (2766, 3)
Scaled feature shape: (2766, 3)

Scaled feature means:
[-0.  0. -0.]

Scaled feature standard deviations:
[1. 1. 1.]

First 5 scaled rows:
[[0.6723 1.6762 1.7878]
 [0.6445 1.6211 1.7841]
 [0.6167 1.5661 1.7804]
 [0.589  1.511  1.7767]
 [0.5612 1.456  1.773 ]]

K-Means preparation completed.
Number of samples: 2766
Number of features: 3


,Annual_Erosion_Rate_m_yr,Physical_Erosion_Score,Economic_Score
count,2.766000e+03,2.766000e+03,2.766000e+03
mean,-6.165230e-17,1.644061e-16,-6.165230e-17
std,1.000181e+00,1.000181e+00,1.000181e+00
min,-1.113778e+00,-1.866842e+00,-1.070337e+00
25%,-6.671201e-01,-7.681276e-01,-9.512496e-01
50%,-2.940376e-01,4.671843e-02,-3.836191e-01
75%,7.810479e-02,6.611878e-01,8.350626e-01
max,4.497463e+00,1.676198e+00,2.502287e+00
